# 07 — Avaliação Comparativa Final

Implementa a Fase 18 do plano de elaboração e a Seção 4.9 do documento
mestre: testes estatísticos entre abordagens, um estudo de ablação e as
visualizações finais para a dissertação.

**Escopo da comparação estatística estrita** (McNemar/Friedman/Nemenyi
exigem as *mesmas* instâncias avaliadas por todos os modelos): inclui
apenas LSTM/CNN/BERTimbau/RoBERTa/DistilBERT/LLM, reavaliados aqui sobre
um subconjunto comum de `paths.test_corpus_file`. Os classificadores
clássicos (`notebooks/04_ml_classico.ipynb`) foram avaliados em uma
partição interna de holdout diferente (ver docstring daquele notebook —
limitação da transformação TF-IDF fora do vocabulário de treino) e por
isso entram apenas na tabela informativa, não nos testes pareados.

**Pré-requisito**: `notebooks/04_ml_classico.ipynb`,
`05_deep_learning_transformers.ipynb` e `06_llm_prompting.ipynb` já
devem ter sido executados (checkpoints em `paths.models_checkpoints_dir`
e relatórios em `paths.reports_metrics_dir`).

In [ ]:
import sys
from pathlib import Path

sys.path.insert(0, str(Path.cwd().parent / "src"))

import numpy as np
import polars as pl

from config.constants import CONFIG_FILE_NAMES
from config.paths import CONFIGS_DIR, load_project_paths
from data.loader import load_training_example_dataset, read_dataset_file
from io_utils.yaml import read_yaml
from models.persistence import load_classifier
from pipelines.comparative_evaluation import run_comparative_evaluation_stage
from pipelines.training_deep_learning import DEFAULT_DEEP_LEARNING_MODEL_NAMES
from visualization.theme import apply_project_theme, save_figure

apply_project_theme()
paths = load_project_paths()
evaluation_config = read_yaml(CONFIGS_DIR / CONFIG_FILE_NAMES["evaluation"])
llm_config = read_yaml(CONFIGS_DIR / CONFIG_FILE_NAMES["llm"])

test_corpus = load_training_example_dataset(paths.test_corpus_file)
# Mesma amostra/semente de `notebooks/06_llm_prompting.ipynb`, para que
# todos os modelos comparados avaliem exatamente as mesmas instâncias.
test_sample = test_corpus.sample(n=min(200, test_corpus.height), seed=42)
y_true = test_sample["sentiment_label"].to_list()

## Reúne predições de LSTM/CNN/Transformers e LLM sobre o subconjunto comum

In [ ]:
model_predictions: dict[str, list[str]] = {}
for model_name in DEFAULT_DEEP_LEARNING_MODEL_NAMES:
    checkpoint_path = paths.models_checkpoints_dir / f"{model_name}.pt"
    model = load_classifier(checkpoint_path, backend="torch")
    model_predictions[model_name] = list(model.predict(test_sample["text"].to_list()))

from pipelines.llm_evaluation import run_llm_evaluation_stage

llm_predictions, _ = run_llm_evaluation_stage(
    test_sample,
    backend_name=next(name for name, config in llm_config["backends"].items() if config["enabled"]),
    strategy=llm_config["prompting"]["default_strategy"],
    prompt_version=llm_config["orchestration"]["prompt_template_version"],
)
# `run_llm_evaluation_stage` não garante a mesma ordem de linhas de
# `test_sample`: reordena pelo id antes de extrair a lista de rótulos.
llm_predictions_ordered = llm_predictions.join(test_sample.select("id"), on="id", how="right")
model_predictions["llm"] = llm_predictions_ordered["sentiment_label"].to_list()

## Avaliação comparativa: significância estatística e fatias

A fatia `comprimento_texto` (curto/longo, mediana do subconjunto) segue
`configs/evaluation.yaml -> slice_evaluation.slices`.

In [ ]:
text_lengths = np.array([len(text) for text in test_sample["text"].to_list()])
length_median = float(np.median(text_lengths))
slice_labels = ["longo" if length > length_median else "curto" for length in text_lengths]

comparative_result = run_comparative_evaluation_stage(
    model_predictions,
    y_true,
    slice_labels=slice_labels,
    output_path=paths.reports_metrics_dir / "comparativo_modelos.csv",
)
print("Teste de Friedman:", comparative_result.friedman_test)
print("Post-hoc de Nemenyi:", comparative_result.nemenyi_test)
comparative_result.merged_report

In [ ]:
for slice_name, slice_report in comparative_result.slice_reports.items():
    print(f"--- fatia: {slice_name} ---")
    display(slice_report)

## Tabela informativa incluindo os classificadores clássicos

Junta os relatórios já salvos por cada notebook anterior — apenas para
leitura lado a lado; os classificadores clássicos não participam dos
testes pareados acima (amostra de avaliação diferente, ver introdução).

In [ ]:
all_reports_paths = {
    "ml_classico": paths.reports_metrics_dir / "avaliacao_ml_classico.csv",
    "deep_learning_transformers": paths.reports_metrics_dir
    / "avaliacao_deep_learning_transformers.csv",
    "llm_prompting": paths.reports_metrics_dir / "avaliacao_llm_prompting.csv",
}
for report_name, report_path in all_reports_paths.items():
    if report_path.exists():
        display(read_dataset_file(report_path))
    else:
        print(f"Relatório '{report_name}' ainda não encontrado em '{report_path}'.")

## Estudo de ablação: pré-processamento de emojis

Demonstra `evaluation.ablation.calculate_ablation_impact` sobre o
componente `"sem_preprocessamento_emojis"` (`configs/evaluation.yaml ->
ablation.components`): compara um classificador logístico sobre TF-IDF
com e sem o token semântico de polaridade de emoji
(`preprocessing.emojis.normalize_emojis` vs. `remove_emojis`), mantendo
todas as demais etapas de normalização idênticas.

In [ ]:
from functools import partial

from data.loader import load_labeled_corpus, load_raw_tweet_dataset
from data.sampler import sample_stratified_subset
from data.splitter import create_stratified_split
from evaluation.ablation import calculate_ablation_impact, identify_most_impactful_component
from evaluation.evaluator import evaluate_classifier
from features.lexical import compute_tfidf_features, pivot_tfidf_features_to_wide
from models.factory import create_classifier
from preprocessing.cleaning import clean_tweet_text
from preprocessing.emojis import normalize_emojis, remove_emojis
from preprocessing.text import (
    normalize_hashtags,
    normalize_mentions,
    normalize_repeated_characters,
    normalize_urls,
)
from training.trainer import Trainer
from utils.text import normalize_whitespace


def _normalize_variant(text: str, *, keep_emoji_signal: bool) -> str:
    """Recompõe `preprocessing.pipeline.normalize_tweet_text`, variando só o passo de emojis."""
    normalized = clean_tweet_text(text)
    normalized = normalize_urls(normalized)
    normalized = normalize_mentions(normalized)
    normalized = normalize_hashtags(normalized)
    normalized = normalize_emojis(normalized) if keep_emoji_signal else remove_emojis(normalized)
    normalized = normalize_repeated_characters(normalized)
    return normalize_whitespace(normalized)


raw_corpus = load_raw_tweet_dataset(paths.raw_tweets_file)
labeled_corpus = load_labeled_corpus(paths.labeled_corpus_file)
ablation_corpus = raw_corpus.join(labeled_corpus.select(["id", "sentiment_label"]), on="id")
ablation_sample = sample_stratified_subset(
    ablation_corpus, stratify_column="sentiment_label", sample_size=min(500, ablation_corpus.height)
)

ablation_metrics: dict[str, dict[str, float]] = {}
for variant_name, keep_emoji_signal in (
    ("com_emojis", True),
    ("sem_preprocessamento_emojis", False),
):
    variant_texts = [
        _normalize_variant(text, keep_emoji_signal=keep_emoji_signal)
        for text in ablation_sample["text"].to_list()
    ]
    variant_corpus = ablation_sample.with_columns(pl.Series("text", variant_texts))
    tfidf_wide = pivot_tfidf_features_to_wide(compute_tfidf_features(variant_corpus))
    joined = variant_corpus.join(tfidf_wide, on="id").sort("id")
    joined_split = create_stratified_split(
        joined, label_column="sentiment_label", test_size=0.2, validation_size=0.0, random_seed=42
    )
    feature_columns = [column for column in tfidf_wide.columns if column != "id"]
    inner_train = joined_split.filter(joined_split["split"] == "treino")
    inner_holdout = joined_split.filter(joined_split["split"] != "treino")

    trainer = Trainer(partial(create_classifier, "logistic_regression"))
    training_result = trainer.fit(
        inner_train.select(feature_columns).to_numpy(), inner_train["sentiment_label"].to_list()
    )
    variant_predictions = training_result.model.predict(
        inner_holdout.select(feature_columns).to_numpy()
    )
    variant_evaluation = evaluate_classifier(
        inner_holdout["sentiment_label"].to_list(), variant_predictions
    )
    ablation_metrics[variant_name] = variant_evaluation.point_metrics

baseline_metrics = ablation_metrics.pop("com_emojis")
ablation_impact = calculate_ablation_impact(
    baseline_metrics, ablation_metrics, metric_name="f1_macro"
)
print(identify_most_impactful_component(ablation_impact))
ablation_impact

## Calibração do melhor modelo

Seleciona o modelo com maior F1-macro na tabela comparativa estrita e
avalia se suas probabilidades são confiáveis o bastante para uso em
decisões automatizadas (`configs/evaluation.yaml -> calibration`).

In [ ]:
from evaluation.calibration import calculate_expected_calibration_error, calculate_reliability_curve
from visualization.diagnostics import plot_calibration_curve

best_model_name = (
    comparative_result.merged_report.filter(pl.col("metric_name") == "f1_macro")
    .sort("metric_value", descending=True)
    .head(1)["model_name"]
    .item()
)
best_model = load_classifier(
    paths.models_checkpoints_dir / f"{best_model_name}.pt", backend="torch"
)
best_model_predictions = best_model.predict(test_sample["text"].to_list())
best_model_scores = best_model.predict_proba(test_sample["text"].to_list())
best_model_confidences = best_model_scores.max(axis=1)

reliability_curve = calculate_reliability_curve(
    y_true,
    best_model_predictions,
    best_model_confidences,
    n_bins=evaluation_config["calibration"]["n_bins"],
)
expected_calibration_error = calculate_expected_calibration_error(
    y_true,
    best_model_predictions,
    best_model_confidences,
    n_bins=evaluation_config["calibration"]["n_bins"],
)
print(f"{best_model_name}: erro de calibração esperado (ECE) = {expected_calibration_error:.4f}")

calibration_figure = plot_calibration_curve(
    reliability_curve, title=f"Calibração — {best_model_name}"
)
save_figure(
    calibration_figure, f"curva_calibracao_{best_model_name}", directory=paths.reports_figures_dir
)

## Conclusões

Registrar aqui, para a redação da dissertação: (1) se o teste de Friedman
rejeita a hipótese de igualdade entre os modelos e quais pares o post-hoc
de Nemenyi confirma como significativamente diferentes; (2) se alguma
fatia (`comprimento_texto`) concentra a maior parte dos erros — um sinal
de que o modelo vencedor não é uniformemente robusto; (3) se remover o
sinal de polaridade de emojis (`sem_preprocessamento_emojis`) tem impacto
prático relevante no F1-macro, e se isso muda a prioridade de outros
componentes de `configs/evaluation.yaml -> ablation.components`
(`sem_embeddings_contextuais`, `sem_autoencoder`, `sem_chain_of_thought`)
ainda não avaliados; (4) se o ECE do melhor modelo é baixo o bastante
para confiar em suas probabilidades em uma eventual predição seletiva
(`metrics.confidence.calculate_selective_prediction_accuracy`).